In [ ]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.riql import RIQLAgent
from src.train_eval import (
    eval_policy_on_env,
    train_riql_from_loader,
    save_metrics_json,
)

In [ ]:
METHOD = "true_only_riql"

SEED_TAG = f"seed_{SEED}"

CKPT_DIR = CHECKPOINTS_DIR / METHOD / ENV_NAME / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / SEED_TAG
OBS_DIR = OBS_STATS_DIR / METHOD / ENV_NAME / SEED_TAG

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("CKPT_DIR:", CKPT_DIR)

In [ ]:
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=0,
    noise_scale=0.0,
    seed=SEED,
    use_timeouts=True,
    noise_type="concat",
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM = true_state_dim  # no encoder, use clean state directly

np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)

print("state_dim:", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim:", LATENT_DIM)
print("Saved obs stats:", OBS_DIR / "obs_stats.npz")

In [ ]:
agent = RIQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
    n_critics=10,
    quantile=0.25,
    huber_delta=1.0,
)

riql_history = train_riql_from_loader(
    riql=agent,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=None,
    repr_mode="true_only",
    use_tqdm=False,
)

In [ ]:
print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=agent,
    env_name=ENV_NAME,
    encoder=None,
    method="true_only",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=0,
    noise_scale=0.0,
    noise_type="concat",
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    extra={
        "latent_dim": LATENT_DIM,
        "riql_epochs": EPOCHS,
        "riql_history": riql_history,
    },
)

print("Saved metrics:", metrics_path)
print(metrics)